In [1]:
!pip install -q langchain langchain-chroma langchain-huggingface sentence-transformers huggingface_hub langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [2]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:

# 1. CONFIGURATION
ACTUAL_DATASET_PATH = "/content/drive/MyDrive/Major_project/Finetuning/MID.json"
REFUSAL_DATASET_PATH = "/content/drive/MyDrive/Major_project/Finetuning/updated_data_refusal.json"

PERSIST_DIR = "./2VectorDatabaseChroma"
COLLECTION_NAME = "gedsi_collection"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

In [4]:
import json
import os
import zipfile
from pathlib import Path


In [5]:

# 2. LOAD & FILTER JSON
file_path=ACTUAL_DATASET_PATH
with open(file_path, 'r', encoding='utf-8') as f:
      data = json.load(f)


In [6]:
len(data)

3399

In [7]:
type(data)

list

In [8]:
data[0]

{'question': 'I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?',
 'answer': '**Yes. Such discriminatory treatment is illegal.**\n\n**Law says:**\n- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."\n\n**Punishment:**\n- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both t

In [9]:
type(data[0])

dict

In [10]:
data[0]["question"]

'I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?'

In [11]:
data[0]["answer"]

'**Yes. Such discriminatory treatment is illegal.**\n\n**Law says:**\n- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."\n\n**Punishment:**\n- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences."\n\n**What to do:**\n1. Collect evidence: note the officer’s name, date, and any witnesses; if possible, record the statement secretly.\n2. File a written complaint at the concerned 

In [13]:
if "question" in data[0] and "answer" in data[0]:
  print('True')

True


In [14]:
def load_and_filter_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    filtered = [item for item in data if 'question' in item and 'answer' in item]
    print(f"Loaded {len(data)} from {Path(file_path).name}, kept {len(filtered)} valid.")
    return filtered

actual_qa = load_and_filter_json(ACTUAL_DATASET_PATH)
refusal_qa = load_and_filter_json(REFUSAL_DATASET_PATH)

Loaded 3399 from MID.json, kept 3324 valid.
Loaded 330 from updated_data_refusal.json, kept 330 valid.


In [15]:
3324+330

3654

In [16]:
# 3. CREATE DOCUMENTS (Question only + Answer in metadata)
def create_question_only_documents(qa_list, doc_type):
    docs = []
    for item in qa_list:
        doc = Document(
            page_content=item['question'],      # only question is embedded
            metadata={
                "type": doc_type,
                "question": item['question'],
                "answer": item['answer']
            }
        )
        docs.append(doc)
    return docs

actual_docs = create_question_only_documents(actual_qa, "actual")
refusal_docs = create_question_only_documents(refusal_qa, "refusal")

In [17]:
all_docs = actual_docs + refusal_docs
print(f"Total documents (one per Q&A): {len(all_docs)}")

Total documents (one per Q&A): 3654


In [21]:
type(all_docs)

list

In [20]:
type(all_docs[0])

langchain_core.documents.base.Document

In [18]:
all_docs[0]

Document(metadata={'type': 'actual', 'question': 'I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?', 'answer': '**Yes. Such discriminatory treatment is illegal.**\n\n**Law says:**\n- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."\n\n**Punishment:**\n- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceedi

In [19]:
all_docs[0].page_content

'I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?'

In [22]:
all_docs[0].metadata

{'type': 'actual',
 'question': 'I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?',
 'answer': '**Yes. Such discriminatory treatment is illegal.**\n\n**Law says:**\n- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."\n\n**Punishment:**\n- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousa

In [25]:
type(all_docs[0].metadata)

dict

In [28]:
all_docs[0].metadata.keys()

dict_keys(['type', 'question', 'answer'])

In [24]:
all_docs[0].metadata["type"]

'actual'

In [26]:
all_docs[0].metadata["question"]

'I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?'

In [27]:
all_docs[0].metadata["answer"]

'**Yes. Such discriminatory treatment is illegal.**\n\n**Law says:**\n- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."\n\n**Punishment:**\n- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences."\n\n**What to do:**\n1. Collect evidence: note the officer’s name, date, and any witnesses; if possible, record the statement secretly.\n2. File a written complaint at the concerned 

In [29]:
# 4. SPLITTER
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

split_docs = text_splitter.split_documents(all_docs)
print(f"Chunks after splitting (should equal total documents): {len(split_docs)}")

Chunks after splitting (should equal total documents): 3654


In [30]:
# 5. EMBEDDINGS (GPU + batching)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
# 6. BUILD CHROMA VECTOR STORE
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

In [32]:
print(f"Vector database saved to {PERSIST_DIR}")

Vector database saved to ./2VectorDatabaseChroma


In [33]:
# 7. ZIP & DOWNLOAD
zip_name = "2VectorDatabaseChroma.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(PERSIST_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=os.path.dirname(PERSIST_DIR))
            zipf.write(file_path, arcname)

print(f"Zipped to {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
    print("Download started.")
except ImportError:
    print(f"Manually download {zip_name}.")

Zipped to 2VectorDatabaseChroma.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


In [34]:

# 8. QUICK RETRIEVAL TEST (shows answer from metadata)

print("\n" + "="*60)
print("RETRIEVAL TEST (question-only embedding)")
print("="*60)

test_queries = [
    "I am a woman and the government officer refused to issue me a citizenship certificate.",
    "Hi, how are you today?",
]

for query in test_queries:
    print(f"\n🔍 Query: {query}")
    retrieved = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(retrieved, 1):
        print(f"   Document {i} (type={doc.metadata['type']})")
        print(f"   Question: {doc.metadata['question'][:100]}...")
        print(f"   Answer: {doc.metadata['answer'][:200]}...")
    print("-"*50)

print("\n✅ All done! Your optimised vector database is ready.")


RETRIEVAL TEST (question-only embedding)

🔍 Query: I am a woman and the government officer refused to issue me a citizenship certificate.
   Document 1 (type=actual)
   Question: A government office denied processing my application for a citizenship certificate because of my eth...
   Answer: **No, this is illegal.**

**Law says:**
- The National Penal (Code) Act, 2017, Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise ...
   Document 2 (type=actual)
   Question: I am a person with a disability and a government officer refused to issue my citizenship certificate...
   Answer: **Yes, this is illegal under Nepal law.**

**Law says:**
- Muluki Criminal Code, 2074 Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the ex...
--------------------------------------------------

🔍 Query: Hi, how are you today?
   Document 1 (type=refusal)
   Question: Hi, how are you

In [35]:
# -------------------------------
# 8. QUICK RETRIEVAL TEST (shows answer from metadata)
# -------------------------------
print("\n" + "="*60)
print("RETRIEVAL TEST (question-only embedding)")
print("="*60)

test_queries = [
    "I am a woman and the government officer refused to issue me a citizenship certificate.",
    "Hi, how are you today?",
]

for query in test_queries:
    print(f"\n🔍 Query: {query}")
    retrieved = vectorstore.similarity_search(query, k=1)
    for i, doc in enumerate(retrieved, 1):
        print(f"   Document {i} (type={doc.metadata['type']})")
        print(f"   Question: {doc.metadata['question'][:100]}...")
        print(f"   Answer: {doc.metadata['answer'][:200]}...")
    print("-"*50)

print("\n✅ All done! Your optimised vector database is ready.")


RETRIEVAL TEST (question-only embedding)

🔍 Query: I am a woman and the government officer refused to issue me a citizenship certificate.
   Document 1 (type=actual)
   Question: A government office denied processing my application for a citizenship certificate because of my eth...
   Answer: **No, this is illegal.**

**Law says:**
- The National Penal (Code) Act, 2017, Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise ...
--------------------------------------------------

🔍 Query: Hi, how are you today?
   Document 1 (type=refusal)
   Question: Hi, how are you today?...
   Answer: This information is not available in my training data. I am a specialized chatbot focused only on Gender Equality, Discrimination, and Social Inclusion (GEDSI) topics of Nepal....
--------------------------------------------------

✅ All done! Your optimised vector database is ready.
